# Cosecha masiva de registros OAI

Ejecuta `ListRecords` en modo acotado y permite inspeccionar registros y errores de página antes de guardar el snapshot.


In [ ]:
import inspect
import os
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

import certifi
import pandas as pd
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning


In [ ]:
def get_oai_response(
    base_url,
    verify=None,
    max_retries=3,
    backoff_factor=1.0,
    min_interval=0.0,
    timeout=30.0,
):

    # Usa el bundle de certifi para evitar errores de certificado en requests
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    VERIFY_SSL = os.getenv("OAI_VERIFY_SSL", "false").lower() == "true"
    CA_BUNDLE = os.getenv("OAI_CA_BUNDLE") or certifi.where()
    requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

    verify_param = CA_BUNDLE if VERIFY_SSL else False
    if verify is not None:
        verify_param = verify

    for attempt in range(1, max_retries + 1):
        start_time = time.time()
        response = None
        error = None
        try:
            response = requests.get(base_url, verify=verify_param, timeout=timeout)
        except requests.RequestException as exc:
            error = exc
        elapsed_time = time.time() - start_time

        if min_interval > 0:
            wait_time = max(min_interval - elapsed_time, 0)
            if wait_time > 0:
                print(f"Pausando {wait_time:.2f} segundos para no saturar el servidor")
                time.sleep(wait_time)

        if error:
            print(f"Error en request (intento {attempt}/{max_retries}): {error}")

        if response is not None and response.status_code == 200:
            return response

        status = response.status_code if response is not None else "sin respuesta"
        print(f"Error: {status} (intento {attempt}/{max_retries})")

        if attempt < max_retries:
            backoff = backoff_factor * attempt
            print(f"Reintentando en {backoff:.2f} segundos...")
            time.sleep(backoff)
    return None


In [ ]:
def log_oai_progress(token_elem, total_processed: int):
    """Muestra el avance usando completeListSize y los registros acumulados."""
    if token_elem is None:
        return
    total = token_elem.get('completeListSize')
    try:
        total_int = int(total) if total is not None else None
        if total_int is not None and total_processed is not None:
            remaining = total_int - total_processed
            print(f"Progreso OAI: {total_processed}/{total_int} (faltan ~{remaining})")
    except ValueError:
        # Si el servidor devuelve valores no numéricos, ignora el progreso.
        pass


In [ ]:
OAI_NAMESPACES = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "dc": "http://purl.org/dc/elements/1.1/",
}


In [ ]:
OAI_RECORD_COLUMNS = [
    "record_id", "datestamp", "set_id", "col_id", "title", "date_issued",
    "creators", "description", "types", "identifiers", "languages",
    "subjects", "publishers", "relations", "rights", "formats",
]


In [ ]:
OAI_PROVENANCE_COLUMNS = [
    "_extract_datetime", "_context", "_source_key",
    "_repository_identifier", "_institution_ror", "_base_url",
    "_metadata_prefix",
]


In [ ]:
def add_oai_provenance(
    df: pd.DataFrame,
    *,
    context: str,
    source_key: str,
    repository_identifier: str,
    institution_ror: str,
    base_url: str,
    metadata_prefix: str,
    timestamp: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """Attach the common extraction provenance contract to an OAI dataset."""
    result = df.copy()
    result["_extract_datetime"] = timestamp or pd.Timestamp.now(tz="UTC")
    result["_context"] = context
    result["_source_key"] = source_key
    result["_repository_identifier"] = repository_identifier
    result["_institution_ror"] = institution_ror
    result["_base_url"] = base_url.rstrip("/")
    result["_metadata_prefix"] = metadata_prefix
    return result


In [ ]:
def parse_oai_record(record: ET.Element) -> dict | None:
    """Parse one OAI-PMH record into the canonical raw records schema."""
    header = record.find("oai:header", OAI_NAMESPACES)
    metadata = record.find("oai:metadata", OAI_NAMESPACES)
    if header is None or metadata is None or header.get("status") == "deleted":
        return None

    def _text(path: str):
        node = metadata.find(path, OAI_NAMESPACES)
        return node.text if node is not None else None

    def _texts(path: str) -> list[str | None]:
        return [node.text for node in metadata.findall(path, OAI_NAMESPACES)]

    identifier = header.find("oai:identifier", OAI_NAMESPACES)
    datestamp = header.find("oai:datestamp", OAI_NAMESPACES)
    sets = [node.text for node in header.findall("oai:setSpec", OAI_NAMESPACES)]
    return {
        "record_id": identifier.text if identifier is not None else None,
        "datestamp": datestamp.text if datestamp is not None else None,
        "set_id": sets,
        "col_id": sets[0] if sets else None,
        "title": _text(".//dc:title"),
        "date_issued": _text(".//dc:date"),
        "creators": _texts(".//dc:creator"),
        "description": _texts(".//dc:description"),
        "types": _texts(".//dc:type"),
        "identifiers": _texts(".//dc:identifier"),
        "languages": _texts(".//dc:language"),
        "subjects": _texts(".//dc:subject"),
        "publishers": _texts(".//dc:publisher"),
        "relations": _texts(".//dc:relation"),
        "rights": _texts(".//dc:rights"),
        "formats": _texts(".//dc:format"),
    }


In [ ]:
def oai_extract_records(
    base_url: str,
    context: str,
    env: str,
    source_key: str,
    repository_identifier: str,
    institution_ror: str,
    metadata_prefix: str = "oai_dc",
    dev_page_limit: int = 2,
    initial_resumption_token: str | None = None,
    page_limit: int | None = None,
    date_windows: list[dict[str, str]] | None = None,
    verify=None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if metadata_prefix != "oai_dc":
        raise ValueError(
            "oai_extract_records currently supports metadata_prefix=oai_dc"
        )

    records = []
    page_errors = []

    iteration_limit = dev_page_limit if env == "dev" else page_limit
    windows = date_windows or [{}]
    if initial_resumption_token and len(windows) > 1:
        raise ValueError(
            "initial_resumption_token cannot be combined with multiple date windows"
        )

    for window in windows:
        unknown_keys = set(window) - {"from", "until"}
        if unknown_keys:
            raise ValueError(f"Unsupported OAI date window keys: {unknown_keys}")
        resumption_token = initial_resumption_token
        iteration_count = 0
        window_processed = 0

        while iteration_limit is None or iteration_count < iteration_limit:
            if resumption_token:
                query = {"verb": "ListRecords", "resumptionToken": resumption_token}
            else:
                query = {
                    "verb": "ListRecords",
                    "metadataPrefix": metadata_prefix,
                    **window,
                }
            url = f"{base_url.rstrip('/')}/{context}?{urlencode(query)}"
            print(f"Consultando: {url}")
            response = get_oai_response(url, verify=verify)
            iteration_count += 1

            if response is None or not response.ok:
                if date_windows is None:
                    raise RuntimeError(f"No se pudo completar la cosecha OAI: {url}")
                page_errors.append(
                    {
                        "from": window.get("from"),
                        "until": window.get("until"),
                        "resumption_token": resumption_token,
                        "error_type": "request_failed",
                        "url": url,
                    }
                )
                break

            try:
                root = ET.fromstring(response.text)
            except ET.ParseError as error:
                if date_windows is None:
                    raise RuntimeError(f"Respuesta XML inválida para: {url}") from error
                page_errors.append(
                    {
                        "from": window.get("from"),
                        "until": window.get("until"),
                        "resumption_token": resumption_token,
                        "error_type": "invalid_xml",
                        "url": url,
                    }
                )
                break

            record_nodes = root.findall('.//oai:record', OAI_NAMESPACES)
            for record in record_nodes:
                parsed = parse_oai_record(record)
                if parsed is not None:
                    records.append(parsed)

            window_processed += len(record_nodes)
            token_elem = root.find('.//oai:resumptionToken', OAI_NAMESPACES)
            resumption_token = token_elem.text if token_elem is not None else None
            log_oai_progress(token_elem, window_processed)
            if not resumption_token:
                break

    df = add_oai_provenance(
        pd.DataFrame(records, columns=OAI_RECORD_COLUMNS)
        .drop_duplicates(subset=["record_id"], keep="last")
        .reset_index(drop=True),
        context=context,
        source_key=source_key,
        repository_identifier=repository_identifier,
        institution_ror=institution_ror,
        base_url=base_url,
        metadata_prefix=metadata_prefix,
    )

    errors = add_oai_provenance(
        pd.DataFrame(
            page_errors,
            columns=["from", "until", "resumption_token", "error_type", "url"],
        ),
        context=context,
        source_key=source_key,
        repository_identifier=repository_identifier,
        institution_ror=institution_ror,
        base_url=base_url,
        metadata_prefix=metadata_prefix,
    )
    return df, errors, df.head(100)


In [ ]:
options = catalog.load("params:oai_extract_options").copy()
options["env"] = "dev"
if options.get("date_windows"):
    options["date_windows"] = options["date_windows"][:1]
options


In [ ]:
call_options = {name: options[name] for name in inspect.signature(oai_extract_records).parameters if name in options}
df_records, df_page_errors, df_records_preview = oai_extract_records(**call_options)
assert df_records["record_id"].notna().all()
assert not df_records["record_id"].duplicated().any()
{"records": len(df_records), "page_errors": len(df_page_errors), "sources": df_records["_source_key"].value_counts().to_dict()}


In [ ]:
display(df_records_preview)
display(df_page_errors)
